In [ ]:
from cartopy import crs as ccrs
import matplotlib.pyplot as plt
import owslib
from osgeo import gdal
import geopandas as gpd
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from shapely.geometry import box

proj = ccrs.epsg('25830')
concejo_name = 'Navascues'
concejo_may = 'NAVASCUES'

concejo = gpd.read_file('../BRUTOS/CONCEJOS/Navascues_DEF.shp')
mc = gpd.read_file('../BRUTOS/MC/mc_forestal_3.shp')

concejo_ext = gpd.GeoSeries(
    [
        box(*box(*concejo.total_bounds).buffer(1000).bounds).difference(
            concejo["geometry"].values[0]
        )
    ],
    crs=proj,
)

minx = concejo.geometry.bounds.minx[0]
maxx = concejo.geometry.bounds.maxx[0]
miny = concejo.geometry.bounds.miny[0]
maxy = concejo.geometry.bounds.maxy[0]

fig = plt.figure(figsize=(36,24))
buffer = 1000
ax = plt.axes(projection=proj)
ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)

mc.plot(column='IDCOMBUSTI', cmap='OrRd', ax=ax, legend=True, alpha=0.5, legend_kwds={'fontsize': 20})

ax.add_wms(wms='https://www.ign.es/wms-inspire/mapa-raster', layers=['mtn_rasterizado'])
for i in concejo_ext.geometry:
    ax.add_geometries(i, facecolor=(0,0,0,0.6), edgecolor='blue', linewidth=3, crs=proj)

plt.show()
fig.savefig('../FINALES/MC/{}/mc.png'.format(concejo_may), bbox_inches='tight')

In [ ]:
m = mc.intersection(concejo.geometry[0])

s = m[mc.intersects(concejo.geometry[0]) == True]
r = mc[mc.intersects(concejo.geometry[0]) == True]

r.loc[:, 'geom'] = s
r['area'] = r['geom'].area/10000

r = r[['mc', 'geom', 'area', 'COMBUSTI']]
r = r.set_geometry('geom')
t = r.dissolve(by='mc', aggfunc='sum')
area_total = t['area'].sum()
t['perc'] = t['area'] * 100 / area_total
t = t[['area', 'perc']]
display(t.to_html())

In [ ]:
import os
import geopandas as gpd
from matplotlib import pyplot as plt
from matplotlib.colors import LightSource
from osgeo import gdal
from cartopy import crs as ccrs

proj = ccrs.epsg('25830')
buffer = 1000

# display(redNatura)
ls = LightSource()
ds = gdal.Open('../BRUTOS/MDT/mdt_flammap.asc')

data = ds.ReadAsArray()
gt = ds.GetGeoTransform()

ys, xs = data.shape
ulx, xres, _, uly, _, yres = gt
extent = [ulx, ulx+xres*xs, uly, uly+yres*ys]

path = '../BRUTOS/CONCEJOS/'

for z in os.listdir(path):
    if z.endswith('shp'):
        zona = gpd.read_file(os.path.join(path, z))
        nombre = z.split('_')[0]
        zo = zona.geometry
        mc = gpd.read_file('../BRUTOS/MC/mc_forestal_3.shp')
        
        # print(zo[0])
        mc = mc[mc.geometry.intersects(zo[0]) == True]
        mc['geometry'] = mc.geometry.intersection(zo[0])
        display(mc)
        
        minx, miny, maxx, maxy = zona['geometry'].bounds.values[0]

        fig = plt.figure(figsize=(24,16))
        ax = plt.axes(projection=proj)
        ax.set_extent([minx - buffer, maxx + buffer, miny - buffer, maxy + buffer], crs=proj)
        ax.imshow(ls.hillshade(data), extent=extent, origin='lower', cmap='gray', alpha=0.2)
         # ax.add_wms(wms='https://servicios.idee.es/wms-inspire/mdt', layers=['EL.ElevationGridCoverage'], styles='Elevaciones')
        zona.plot(ax=ax, facecolor=(1,0,0,0.1), edgecolor='black', linewidth=3)
        mc.plot(column='IDCOMBUSTI', ax=ax, cmap='tab20', edgecolor='b', alpha=0.6, legend=True, legend_kwds={'fontsize': 16})
        plt.savefig('../FINALES/MC/mc_{}.png'.format(nombre), bbox_inches='tight')